We'll be using the Ollama-env environment kernel here.

you might be wondering why am I using another jupyter notebook and another kernel (ollama-env) for this?

- well because I want to separate the workflow and I'm following this tutorial to not get any errors. 🙂

---

for the ollama model, you are required to have it downloaded in your system and be recognized by the ollama system. To use it, run 

`ollama run llama3.1:latest`

...it should automatically download the model into your system, which you can then use in this JuPyTer notebook.

In [1]:
import pandas as pd
import ollama

In [ ]:
# SETTINGS
model = "llama3.1:latest"
sheet_name = "main"

# --- THE FUNCTIONS ---

inputDF = pd.read_excel("input-data/main-excel (2).xlsx", sheet_name=sheet_name)

# 1. Strip whitespace and check for empty strings, OR check for NaN
is_blank = (inputDF['area_type'].str.strip() == "") | (inputDF['area_type'].isna())
filteredDF = inputDF[is_blank]
filteredDF = filteredDF[["exact_location", "area_type"]]

print("MAX:", len(inputDF), "|", len(filteredDF), "rows left to process.")

def call_ollama_on_location(location, idx):
    # print("calling ollama for idx:", idx, "| location:", location)
    response = ollama.chat(
        model=model,
        messages=[{
            "role": "user",
            "content": f"Is this rural, suburban or urban. Location: {location}? Answer with only one word: rural, suburban or urban."
        }]
    )
    areaType = response["message"]["content"]
    
    areaType = areaType.lower().strip().capitalize() # cleaning output
    if (areaType.endswith(".") or areaType.endswith(",")):
        areaType = areaType[:-1]
    if (areaType.lower() not in ["rural", "suburban", "urban"]):
        areaType = "" # if the response is not one of the expected values, set it to empty string
    return areaType


def process_area_types(idx, location, myCounter):
    myDict = {
        "area_type": "",
        "exact_location": "",
    }

    areaType = call_ollama_on_location(location, idx)
    print("processed idx:", idx, "|", f"{myCounter}/{len(filteredDF)}", "| area-type:", areaType,"| location:", location)
    myDict["area_type"] = areaType
    myDict["exact_location"] = location
    myDict["idx"] = idx

    return myDict

def process_area_types_v2(idx, location, myCounter):
    myDict = {
        "area_type": "",
        "exact_location": "",
    }
    areaType = call_ollama_on_location(location, idx)
    myDict["area_type"] = areaType
    myDict["exact_location"] = location
    myDict["idx"] = idx

    inputDF.at[idx, "area_type"] = areaType
    # output
    print("processed idx:", idx, "|", f"{myCounter}/{len(filteredDF)}", "| area-type:", areaType,"| location:", location)
    inputDF.to_excel("output-data/main-excel-output.xlsx", sheet_name=sheet_name, index=False) # save after each process.

# --- END OF FUNCTIONS ---

MAX: 4344 | 0 rows left to process.


This is really making my laptop fans spin. But hey, the data is being processed locally and for free and I don't have to make a complicated system to determine if a location is either Urban, Suburban, or Rural.

But I would like to make a disclaimer: The LLM is not an authoritative source, although it can give me a summary of the answer, it is sometimes not true. Currently the answers I'm getting, I'm going to believe its true because this LLM model comes from Meta (company that owns Facebook) which is a reputable company and is also a company who owns a lot of data.

In [3]:
import concurrent.futures as cf
import time

# SETTINGS
max_workers = 20
activate_this_code_block_1 = False

results = []

if activate_this_code_block_1:
    with cf.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        myCounter = 1
        for row in filteredDF.itertuples():
            idx = row.Index
            location = row.exact_location

            print(f"activate worker #{idx}")
            future = executor.submit(process_area_types_v2, idx, location, myCounter)
            futures.append(future)
            myCounter += 1
    print("All tasks submitted. Waiting for completion...")